In [1]:
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

class Residual(nn.Module):
    def __init__(self, input_channels, num_channels, use_1x1conv=False, strides=1):
        super().__init__()

        self.conv1 = nn.Conv2d(
            input_channels,
            num_channels,
            kernel_size=3,
            padding=1,
            stride=strides
        )
        
        self.conv2 = nn.Conv2d(
            num_channels,
            num_channels,
            kernel_size=3,
            padding=1,
        )

        if use_1x1conv:
            self.conv3 = nn.Conv2d(
                input_channels,
                num_channels,
                kernel_size=1,
                stride=strides
            )
        else:
            self.conv3 = None
        
        self.bn1 = nn.BatchNorm2d(num_channels)
        self.bn2 = nn.BatchNorm2d(num_channels)
    
    def forward(self, X):
        Y = F.relu(self.bn1(self.conv1(X)))
        Y = self.bn2(self.bn2(Y))

        if self.conv3:
            X = self.conv3(X)
        
        Y += X
        return F.relu(Y)

In [2]:
b1 = nn.Sequential(
    nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
)

def resent_block(input_channels, num_channels, num_residents, first_block=False):
    blk = []

    for i in range(num_residents):
        if i == 0 and not first_block:
            blk.append(
                Residual(
                    input_channels,
                    num_channels,
                    use_1x1conv=True,
                    strides=2
                )
            )
        else:
            blk.append(
                Residual(
                    num_channels,
                    num_channels,
                )
            )
    
    return blk

b2 = nn.Sequential(*resent_block(64, 64, 2, first_block=True))
b3 = nn.Sequential(*resent_block(64, 128, 2))
b4 = nn.Sequential(*resent_block(128, 256, 2))
b5 = nn.Sequential(*resent_block(256, 512, 2))

net = nn.Sequential(
    b1, b2, b3, b4, b5,
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),
    nn.Linear(512, 10)
)

In [3]:
X = torch.rand(size=(1, 1, 224, 224))
for layer in net:
    X = layer(X)
    print(layer.__class__.__name__, 'shape: ', X.shape)

Sequential shape:  torch.Size([1, 64, 56, 56])
Sequential shape:  torch.Size([1, 64, 56, 56])
Sequential shape:  torch.Size([1, 128, 28, 28])
Sequential shape:  torch.Size([1, 256, 14, 14])
Sequential shape:  torch.Size([1, 512, 7, 7])
AdaptiveAvgPool2d shape:  torch.Size([1, 512, 1, 1])
Flatten shape:  torch.Size([1, 512])
Linear shape:  torch.Size([1, 10])
